# HubbardBath 04 — Liouvillian spectral gap vs thermal mixing time

This notebook is the first parameter-sweep research experiment in HubbardBath. We ask whether the slowest non-stationary Liouvillian decay rate explains the operational time required for a non-equilibrium Hubbard state to remain close to its Gibbs state.

**Question:** How do `U/t` and `βt` change the Liouvillian gap and mixing time, and does `t_mix` approximately track `1/Δ_L` across this finite two-site grid?


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'HubbardBath']
for candidate in candidates:
    if (candidate / 'src' / 'spectral.py').exists():
        repo_root = candidate.resolve()
        break
else:
    raise FileNotFoundError('Run this notebook from the HubbardBath repository.')

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.hubbard import project_to_particle_sector, two_site_hubbard_hamiltonian
from src.open_systems import build_thermal_liouvillian, doublon_initial_state
from src.spectral import (
    liouvillian_spectral_gap,
    loglog_power_law_fit,
    stationary_mode_count,
    thermalisation_metrics,
)


## 1. What is the Liouvillian spectral gap?

For Lindblad evolution `dρ/dt = L(ρ)`, the stationary mode has eigenvalue zero. Non-stationary modes decay according to the real parts of the other eigenvalues. We define

$$\Delta_{\mathcal L}=\min_{\mathrm{Re}(\lambda_j)<0}-\mathrm{Re}(\lambda_j).$$

A smaller gap means a slower asymptotic decay mode. The quantity is spectral; it does not depend on the particular initial state. Our mixing time below *does* depend on the chosen initial doublon state and tolerance.


In [ ]:
U = 4.0
beta = 1.0

H_full = two_site_hubbard_hamiltonian(t=1.0, U=U)
H_half, _ = project_to_particle_sector(H_full, 4, 2)
L, _, rho_beta = build_thermal_liouvillian(H_half, beta=beta, rate_scale=0.2)

gap = liouvillian_spectral_gap(L)
print(f'Liouvillian gap: {gap:.6f}')
print(f'Inverse gap: {1/gap:.6f}')
print('Stationary modes:', stationary_mode_count(L))


## 2. Operational mixing time

For an initial doublon state, define mixing at tolerance `ε = 10^-2` as the first sampled time after which the trace distance stays below `ε` for the remainder of the simulation horizon.


In [ ]:
metrics = thermalisation_metrics(
    L,
    doublon_initial_state(),
    rho_beta,
    epsilon=1e-2,
    horizon_in_inverse_gaps=12.0,
    num_time_points=241,
)
metrics


## 3. Sweep interaction strength and temperature

The default grid contains 30 exact small-system cases. Each point constructs its own thermal Lindbladian, computes the gap, evolves the same initial doublon state, and extracts the persistent-threshold mixing time.


In [ ]:
U_values = [0.0, 1.0, 2.0, 4.0, 6.0, 8.0]
beta_values = [0.25, 0.5, 1.0, 2.0, 4.0]
rho0 = doublon_initial_state()
rows = []

for U_value in U_values:
    H_full = two_site_hubbard_hamiltonian(t=1.0, U=U_value)
    H_half, _ = project_to_particle_sector(H_full, 4, 2)
    for beta_value in beta_values:
        L, _, rho_beta = build_thermal_liouvillian(
            H_half, beta=beta_value, rate_scale=0.2
        )
        m = thermalisation_metrics(
            L, rho0, rho_beta,
            epsilon=1e-2,
            horizon_in_inverse_gaps=12.0,
            num_time_points=241,
        )
        rows.append({'U_over_t': U_value, 'beta_t': beta_value, **m})

len(rows)


## 4. Does mixing time track the inverse gap?

We fit only a descriptive finite-grid power law

$$t_{\mathrm{mix}} = A (1/\Delta_{\mathcal L})^p.$$

This is **not** an asymptotic scaling claim. It is a compact summary of the 30 exact cases above.


In [ ]:
inverse_gap = np.array([row['inverse_gap'] for row in rows])
mixing_time = np.array([row['mixing_time'] for row in rows])
fit = loglog_power_law_fit(inverse_gap, mixing_time)
fit


In [ ]:
order = np.argsort(inverse_gap)
plt.figure(figsize=(7, 4.8))
for beta_value in beta_values:
    mask = np.array([row['beta_t'] == beta_value for row in rows])
    plt.scatter(inverse_gap[mask], mixing_time[mask], label=fr'$\beta t={beta_value}$')

plt.plot(
    inverse_gap[order],
    fit['prefactor'] * inverse_gap[order] ** fit['exponent'],
    linestyle='--',
    label=fr"fit: exponent={fit['exponent']:.3f}, $R^2$={fit['r_squared']:.3f}",
)
plt.xlabel(r'$1/\Delta_{\mathcal L}$')
plt.ylabel(r'$t_{\rm mix}$')
plt.title('Finite-system mixing time versus inverse Liouvillian gap')
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 5. Interpretation checklist

Before making any claim, check:

1. every case has one stationary mode;
2. every trajectory actually reaches the chosen tolerance;
3. the fit is not being driven by one temperature slice;
4. `Δ_L t_mix` is inspected directly;
5. all conclusions are stated as finite-system observations for this bath model.

**Next milestone:** map the same fermionic Hamiltonian into an explicit Pauli-string qubit Hamiltonian and verify spectral equivalence. That begins the quantum-algorithm half of HubbardBath.
